[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [3]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [5]:
import torch
import torch.nn as nn
import math

In [19]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        d_k = d_model // num_heads
        self.num_heads = num_heads
        self.d_model = d_model
        self.num_kv_heads = num_kv_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * d_k)
        self.W_v = nn.Linear(d_model, num_kv_heads * d_k)
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, x): # Self-attention with grouped KV
        B, S, _ = x.shape
        d_k = self.d_model // self.num_heads
        
        num_repeat = self.num_heads // self.num_kv_heads

        
        q = self.W_q(x) # (B, S, d_model)
        q = q.view(B, S, self.num_heads, d_k).transpose(1, 2) # (B, self.num_heads, S, d_k)

    
        k = self.W_k(x).view(B, S, self.num_kv_heads, d_k) # (B, S, self.num_kv_heads, d_k)
        k = k.unsqueeze(3).expand(-1, -1, -1, num_repeat, -1).reshape(B, S, self.num_heads, d_k).transpose(1, 2)
        # or
        # k = k.repeat_interleave(num_repeat, dim=1)
        #  对比：
          # ┌──────────────────────────┬────────────────────┐
          # │           方法           │        代码        │
          # ├──────────────────────────┼────────────────────┤
          # │ unsqueeze+expand+reshape │ 3 步，不分配内存   │
          # ├──────────────────────────┼────────────────────┤
          # │ repeat_interleave        │ 1 步，但会复制内存 │
          # └──────────────────────────┴────────────────────┘

        

        
        v = self.W_v(x).view(B, S, self.num_kv_heads, d_k)
        v = v.unsqueeze(3).expand(-1, -1, -1, num_repeat, -1).reshape(B, S, self.num_heads, d_k).transpose(1, 2)
            
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.d_model ** 0.5)
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v).reshape(B, S, self.d_model)
        return self.W_o(out)



In [20]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [21]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (1.5ms)
  ✅ [2/5] nn.Linear with correct shapes (0.2ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (0.6ms)
  ✅ [4/5] KV heads are shared correctly (0.6ms)
  ✅ [5/5] Gradient flow (19.0ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (21.9ms total)
  Progress saved. Run status() to see your dashboard.

